In [0]:
from pyspark.sql.functions import col, explode, current_timestamp


json_path = "/Volumes/workspace/steam/steam_volume/steam_prices.json"


df_raw = (
    spark.read
    .option("multiline", "true")
    .json(json_path)
)

df_flat = (
    df_raw
    .select(
        col("appID").alias("appid"),
        col("name"),
        col("is_free"),
        col("price_overview.final_formatted").alias("price_final_formatted"),
        col("price_overview.initial_formatted").alias("price_initial_formatted"),
        col("price_overview.discount_percent").alias("discount_percent"),
        col("price_overview.final").alias("price_final_cents"),
        col("price_overview.initial").alias("price_initial_cents"),
        col("price_overview.currency").alias("currency")
    )
)

df_bronze_prices = df_flat.withColumn("ingest_ts", current_timestamp())

bronze_prices_path = "dbfs:/Volumes/workspace/steam/steam_volume/bronze_prices"

df_bronze_prices.write.format("delta").mode("overwrite").save(bronze_prices_path)


display(df_bronze_prices)


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.steam.bronze_prices_df AS
SELECT *
FROM delta.`/Volumes/workspace/steam/steam_volume/bronze_prices`;